# Multimodal product recommendation, trained from scratch (Kaggle GPU)

Trains every model of the study **from random initialisation** on the full 44k-product catalog and produces the
result tables, figures and a zip for the local app.

**Setup (once)**
1. Verify your phone number on Kaggle (*Settings → Phone verification*); this is needed for GPU and internet.
2. Upload `mmrec_code.zip` as a dataset: *Datasets → New Dataset → upload the zip → name it `mmrec-code` → Create*.
3. In this notebook's right-hand panel: **Accelerator: GPU T4 x2** (or P100), **Internet: On**,
   **Add Input → Your Datasets → mmrec-code**.

**Run:** click **Save Version → Save & Run All (Commit) → Save**. It runs in the background for about 3–4 hours,
so you can close the browser. Afterwards, open the version, go to **Output**, and download
**`mmrec_scratch_results.zip`**.

In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "No GPU: right panel > Session options > Accelerator > GPU T4 x2 (or P100)"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Find the project code among the notebook inputs (Kaggle unpacks uploaded zips automatically)
import os, shutil, zipfile, pathlib
PROJ = pathlib.Path("/kaggle/working/mmrec")
if PROJ.exists():
    shutil.rmtree(PROJ)
src_dir = next((p.parent.parent.parent for p in pathlib.Path("/kaggle/input").rglob("experiments.py")
                if p.parent.name == "scratch"), None)
if src_dir is not None:
    shutil.copytree(src_dir, PROJ)
else:
    z = next(pathlib.Path("/kaggle/input").rglob("mmrec_code.zip"), None)
    assert z is not None, "Add the mmrec-code dataset as input (right panel > Add Input)"
    zipfile.ZipFile(z).extractall(PROJ)
os.chdir(PROJ)
print(sorted(p.name for p in PROJ.iterdir()))

In [ ]:
# 3. Libraries (needs Internet: On)
!pip -q install datasets

In [ ]:
# 4. Dataset: full 44k catalog from Hugging Face -> 64x64 images, tokenizer, 70/15/15 split
!python -m src.scratch.data --source hf
import pandas as pd
m = pd.read_csv("data/scratch/meta.csv")
print(len(m), "products,", m.articleType.nunique(), "article types"); print(m.split.value_counts())

In [ ]:
# 5. The full study: optimizer tuning -> SGD/Adam/AdamW -> 6 methods -> ablations (about 3-4 hours)
OUT = "/kaggle/working/out"
!python -m src.scratch.experiments --out {OUT} --stages tune optimizers methods ablations

In [ ]:
# 6. Tables, significance tests, figures and the export for the local app
!python -m src.scratch.experiments --out {OUT} --stages report export --bundle /kaggle/working/mmrec_scratch_results.zip

In [ ]:
# 7. Results
from IPython.display import Image, display
res = pathlib.Path(OUT) / "results" / "scratch"
print(pd.read_csv(res / "method_comparison.csv").round(3).to_string(index=False))
print(pd.read_csv(res / "optimizer_comparison.csv").round(3).to_string(index=False))
print(pd.read_csv(res / "ablations.csv").round(3).to_string(index=False))
for f in ["method_comparison.png", "optimizer_curves.png", "optimizer_comparison.png", "ablations.png"]:
    display(Image(filename=str(res / f)))

In [ ]:
# 8. Keep the Output tab small: remove the large training arrays (the zip has everything the app needs)
for f in ["data/scratch/images.npy", "data/scratch/tokens.npy"]:
    pathlib.Path(f).unlink(missing_ok=True)
print("Download mmrec_scratch_results.zip from the Output tab:",
      round(os.path.getsize("/kaggle/working/mmrec_scratch_results.zip") / 2**20, 1), "MB")